# 1. Imports

In [1]:
import os
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 2. Config

In [12]:
ROOT_PATH = Path("E:\WLASL\wlasl_1000_preproc")

SAVE_FOLDER = Path("wlasl1000_1")

VIDEO_NPY_PATH = ROOT_PATH / "videos"

TRAIN_JSON = ROOT_PATH / "train_final.json"
VAL_JSON = ROOT_PATH / "val_final.json"
TEST_JSON = ROOT_PATH / "test_final.json"

LABEL_MAP_PATH = ROOT_PATH / "label_map_final.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_LEN = 150   # truncate/pad sequence length
BATCH_SIZE = 32

<>:1: SyntaxWarning: invalid escape sequence '\W'
<>:1: SyntaxWarning: invalid escape sequence '\W'
C:\Users\tahmi\AppData\Local\Temp\ipykernel_28712\2939858455.py:1: SyntaxWarning: invalid escape sequence '\W'
  ROOT_PATH = Path("E:\WLASL\wlasl_1000_preproc")


In [14]:
paths = {
    "VIDEO_NPY_PATH": VIDEO_NPY_PATH,
    "TRAIN_JSON": TRAIN_JSON,
    "VAL_JSON": VAL_JSON,
    "TEST_JSON": TEST_JSON,
    "LABEL_MAP_PATH": LABEL_MAP_PATH,
    "SAVE FOLDER": SAVE_FOLDER,
}

for name, path in paths.items():
    if path.exists():
        print(f"{name} exists at {path}")
    else:
        print(f"{name} does NOT exist at {path}")

VIDEO_NPY_PATH exists at E:\WLASL\wlasl_1000_preproc\videos
TRAIN_JSON exists at E:\WLASL\wlasl_1000_preproc\train_final.json
VAL_JSON exists at E:\WLASL\wlasl_1000_preproc\val_final.json
TEST_JSON exists at E:\WLASL\wlasl_1000_preproc\test_final.json
LABEL_MAP_PATH exists at E:\WLASL\wlasl_1000_preproc\label_map_final.json
SAVE FOLDER exists at wlasl1000_1


# 3. Load Label Map

In [9]:
with open(LABEL_MAP_PATH) as f:
    label_map = json.load(f)

# Reverse mapping: gloss -> label index
gloss_to_idx = {v: int(k)-1 for k, v in label_map.items()}
idx_to_gloss = {int(k)-1: v for k, v in label_map.items()}

NUM_CLASSES = len(gloss_to_idx)

print("Total classes:", NUM_CLASSES)

Total classes: 1000


# 4. JSON Parser

vid_id --> Label

In [10]:
def parse_split(json_path):
    with open(json_path) as f:
        data = json.load(f)

    samples = []

    for item in data:
        gloss = item["gloss"]

        if gloss not in gloss_to_idx:
            continue

        label = gloss_to_idx[gloss]

        for inst in item["instances"]:
            vid = inst["video_id"]
            samples.append((vid, label))

    return samples

### Build vidId -> Label Map

In [16]:
import pickle
from pathlib import Path
from tqdm import tqdm

def build_index(npy_root, save_folder=None, filename="video_index.pkl"):
    save_path = None
    if save_folder is not None:
        save_path = Path(save_folder) / filename

        if save_path.exists():
            with open(save_path, "rb") as f:
                print("Reusing saved index...")
                return pickle.load(f)


    index = {}
    for folder in tqdm(Path(npy_root).iterdir()):
        if not folder.is_dir():
            continue
        for file in folder.glob("*.npy"):
            vid = file.stem
            index[vid] = file


    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with open(save_path, "wb") as f:
            pickle.dump(index, f)
        print(f"Index saved to {save_path}")

    return index


video_index = build_index(VIDEO_NPY_PATH, SAVE_FOLDER)
print("Indexed videos:", len(video_index))

Reusing saved index...
Indexed videos: 6073


# 5. Dataset

In [17]:
class SignDataset(Dataset):
    def __init__(self, samples, video_index, max_len=150):
        self.samples = samples
        self.video_index = video_index
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vid, label = self.samples[idx]

        path = self.video_index.get(vid, None)

        if path is None:
            # fallback empty tensor
            data = np.zeros((self.max_len, 75, 3), dtype=np.float32)
        else:
            data = np.load(path)  # (T,75,3)

        T = data.shape[0]

        # Pad / truncate
        if T > self.max_len:
            data = data[:self.max_len]
        else:
            pad = np.zeros((self.max_len - T, 75, 3))
            data = np.concatenate([data, pad], axis=0)

        data = torch.tensor(data, dtype=torch.float32)

        # reshape → (T, features)
        data = data.view(self.max_len, -1)  # (T, 225)

        return data, label

# 6. Dataloaders

In [ ]:
train_ds = SignDataset(train_samples, video_index, MAX_LEN)
val_ds = SignDataset(val_samples, video_index, MAX_LEN)
test_ds = SignDataset(test_samples, video_index, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)